# Subsidiary Data Analysis
Analyze parsed subsidiary data from SEC filings

## 0. Setup

In [124]:
import polars as pl
from pathlib import Path

# Load CSV
csv_path = Path("../../../../subsidiaries_SUCCESS.csv")
df = pl.read_csv(csv_path, schema_overrides={"Ownership": pl.Float64})

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns}")
df.head()

Total rows: 180,477
Columns: ['Accession', 'URL', 'SubsidiaryId', 'Subsidiary', 'Jurisdiction', 'NestingLevel', 'ParentName', 'ParentId', 'Ownership', 'Footnotes']


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""4d83702c-06da-50c0-ac82-06fb9e…","""Medallion Funding LLC""","""New York""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",null,""""""
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""19586335-b58e-51a7-83d2-a416e9…","""Medallion Capital, Inc.""","""Minnesota""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",null,""""""
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""3f195423-2705-5a6d-ae1a-c9fe58…","""Freshstart Venture Capital Cor…","""New York""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",null,""""""
"""`000095017025038693""","""https://www.sec.gov/Archives/e…","""ad03fb59-b4f4-5097-aa3c-ac60d4…","""Medallion Bank""","""Utah""",0,null,"""6bd69361-5068-5121-9d53-9dd804…",null,""""""
"""`000100022825000014""","""https://www.sec.gov/Archives/e…","""2eb8ef76-d45d-59f7-891c-9af211…","""ACE Surgical Supply Co., Inc.""","""Massachusetts""",0,null,"""0644a2d7-c768-5a0d-aca8-83406d…",null,""""""


In [125]:
# Basic stats
df.describe()

statistic,Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,str,f64,str,str,f64,str
"""count""","""180477""","""180477""","""180477""","""180477""","""180477""",180477.0,"""3459""","""180477""",6926.0,"""180477"""
"""null_count""","""0""","""0""","""0""","""0""","""0""",0.0,"""177018""","""0""",173551.0,"""0"""
"""mean""",null,null,null,null,null,0.055769,null,null,94.504417,null
"""std""",null,null,null,null,null,0.363995,null,null,17.392009,null
"""min""","""`000000248825000012""","""https://www.sec.gov/Archives/e…","""0000f001-2d22-53b0-bd26-620761…","""""Abbott Laboratories Baltics""""","""""",0.0,"""1. United Rentals Highway Tech…","""002b5809-de06-5c7c-a24e-82476e…",0.0,""""""
"""25%""",null,null,null,null,null,0.0,null,null,100.0,null
"""50%""",null,null,null,null,null,0.0,null,null,100.0,null
"""75%""",null,null,null,null,null,0.0,null,null,100.0,null
"""max""","""`000207709625000107""","""https://www.sec.gov/Archives/e…","""ffffbca8-b134-5c99-833d-969fe2…","""深圳前海豐泰仁匯健康科技有限公司""","""•Luxembourg""",8.0,"""◦Kanawha River Terminals LLC""","""fff74162-5a60-5611-b1c4-eafb04…",100.0,"""99"""


## 1. Rows with Empty Jurisdiction

In [134]:
# Query rows for a specific accession
accession_to_find = "`000162828025008991"
df.filter(pl.col("Accession") == accession_to_find).head(10)

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""36b299b7-678c-536f-b13c-992b65…","""Provident Financial Services, …","""New Jersey""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""ce6e288c-0184-580b-841f-ccd0d9…","""Sussex Capital Trust II""","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""9d3998d0-09ca-5e4d-9357-8237d7…","""1st Constitution Capital Trust…","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""e42c3014-7a80-5170-9175-62a121…","""Lakeland Bancorp Capital Trust…","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""38925c3a-f5e2-5f4c-9331-406f11…","""Lakeland Bancorp Capital Trust…","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""


In [137]:
# Find rows with empty or null jurisdiction
empty_jurisdiction = df.filter(
    pl.col("Jurisdiction").is_null() | (pl.col("Jurisdiction").str.strip_chars() == "")
)

unique_accessions = empty_jurisdiction.select("Accession").unique()
print(f"Rows with empty jurisdiction: {len(empty_jurisdiction):,}")

print(f"\nUnique accession list({len(unique_accessions)}): ")
print(unique_accessions.to_series().to_list())
empty_jurisdiction.head(10)

Rows with empty jurisdiction: 1,526

Unique accession list(34): 
['`000007754325000025', '`000141210025000011', '`000181949325000043', '`000153592925000017', '`000110465925121185', '`000162828025012639', '`000138282125000046', '`000110465925123342', '`000110465925123106', '`000035069825000029', '`000162828025014751', '`000162828025008991', '`000162828025008656', '`000143774925005487', '`000035603725000065', '`000151470525000004', '`000141057825000855', '`000009602125000099', '`000004098725000026', '`000119312525185641', '`000170160525000035', '`000149090625000033', '`000121390025103497', '`000121390025035604', '`000114036125006750', '`000005849225000136', '`000162828025008977', '`000106299325003695', '`000081201125000104', '`000143774925017409', '`000009144025000010', '`000119312525282583', '`000183437625000062', '`000155837025001937']


Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000143774925017409""","""https://www.sec.gov/Archives/e…","""5a593c1c-5a02-5f4e-98fe-d1d6bc…","""SPAR DSI Human Resource Compan…","""""",0,null,"""a24344c5-dda5-5c10-87c8-7fbb80…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""ce6e288c-0184-580b-841f-ccd0d9…","""Sussex Capital Trust II""","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""9d3998d0-09ca-5e4d-9357-8237d7…","""1st Constitution Capital Trust…","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""e42c3014-7a80-5170-9175-62a121…","""Lakeland Bancorp Capital Trust…","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000162828025008991""","""https://www.sec.gov/Archives/e…","""38925c3a-f5e2-5f4c-9331-406f11…","""Lakeland Bancorp Capital Trust…","""""",0,null,"""6211cfe6-9e02-5941-a2bb-1c8107…",null,""""""
"""`000110465925121185""","""https://www.sec.gov/Archives/e…","""37f1337d-1395-58be-bca0-2aff1d…","""West Virginia Pipeline, Inc.""","""""",0,null,"""808d752d-eb79-5884-ad73-a92dc6…",null,""""""
"""`000110465925121185""","""https://www.sec.gov/Archives/e…","""fd4bd264-5b38-53ca-aaa6-cfe88e…","""SQP Construction Group, Inc.""","""""",0,null,"""808d752d-eb79-5884-ad73-a92dc6…",null,""""""
"""`000110465925121185""","""https://www.sec.gov/Archives/e…","""ef229053-6f44-542a-8701-ad92b7…","""TriState Paving & Sealcoating,…","""""",0,null,"""808d752d-eb79-5884-ad73-a92dc6…",null,""""""
"""`000110465925121185""","""https://www.sec.gov/Archives/e…","""8a0f6740-f7e7-5270-93f0-02fdd1…","""Ryan Construction Services, In…","""""",0,null,"""808d752d-eb79-5884-ad73-a92dc6…",null,""""""


## 2. Nested Subsidiaries (NestingLevel >= 1)

In [128]:
# Find nested subsidiaries
nested = df.filter(pl.col("NestingLevel") >= 1)

nested_accessions = nested.select("Accession").unique()
print(f"Nested subsidiaries (level >= 1): {len(nested):,}")

print(f"Unique accession list({len(nested_accessions)}):")
print(nested_accessions.to_series().to_list())
# nested.head(20)


# Filter by specific accession
pl.Config.set_tbl_rows(-1) 
target_accession = '`000007889025000059'
df.filter(pl.col("Accession") == target_accession)


Nested subsidiaries (level >= 1): 6,119
Unique accession list(188):
['`000095017025029095', '`000163279025000091', '`000131916125000034', '`000162828025010947', '`000162828025008656', '`000129281425001352', '`000116486325000009', '`000100291025000055', '`000194136525000016', '`000001961725000270', '`000121390025027319', '`000121390025030345', '`000162828025014348', '`000131415225000031', '`000119312525319187', '`000170569625000033', '`000144889325000009', '`000133152025000076', '`000181428725000006', '`000196891525000006', '`000085120525000012', '`000107997325000827', '`000186919825000009', '`000192956125000044', '`000162828025012095', '`000165785325000015', '`000092480525000012', '`000070895525000012', '`000141626525000006', '`000103130825000002', '`000111316925000007', '`000162828025009448', '`000109269925000011', '`000095017025045214', '`000021546625000009', '`000162828025018102', '`000007528825000033', '`000006638225000069', '`000162828025005392', '`000071742325000006', '`000095017

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""6ac90f06-3307-5758-8504-e3a72d…","""The Pittston Company""","""Delaware""",0,null,"""6efe832d-6b6b-5802-a8fb-92caa2…",null,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""ddddbb31-6aec-53b4-ac81-899083…","""Glen Allen Development, Inc.""","""Delaware""",0,null,"""6efe832d-6b6b-5802-a8fb-92caa2…",null,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""1fb30527-3e37-5d5d-ae97-778d4a…","""Liberty National Development C…","""Delaware""",1,"""Glen Allen Development, Inc.""","""ddddbb31-6aec-53b4-ac81-899083…",32.5,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""a99e06bf-812b-5e5a-bf91-e6a200…","""New Liberty Residential Urban …","""New Jersey""",1,"""Glen Allen Development, Inc.""","""ddddbb31-6aec-53b4-ac81-899083…",17.5,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""6542ddd3-b926-5b52-bc51-015c3b…","""Pittston Services Group Inc.""","""Virginia""",0,null,"""6efe832d-6b6b-5802-a8fb-92caa2…",null,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""974e8b00-dda9-5cba-89cf-c4b6e2…","""Brink’s Holding Company""","""Delaware""",1,"""Pittston Services Group Inc.""","""6542ddd3-b926-5b52-bc51-015c3b…",null,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""6e4060d9-2d82-5be3-8cc4-5a6ccc…","""Brink’s Finance Holding Compan…","""Delaware""",2,"""Brink’s Holding Company""","""974e8b00-dda9-5cba-89cf-c4b6e2…",null,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""de8164bb-3bc0-5a93-bc22-fca9f8…","""Brink’s Capital Holding Compan…","""Delaware""",3,"""Brink’s Finance Holding Compan…","""6e4060d9-2d82-5be3-8cc4-5a6ccc…",null,""""""
"""`000007889025000059""","""https://www.sec.gov/Archives/e…","""bfe1b27a-9612-54f2-b02e-589e53…","""Brink’s Capital, LLC""","""Delaware""",4,"""Brink’s Capital Holding Compan…","""de8164bb-3bc0-5a93-bc22-fca9f8…",null,""""""


In [129]:
# Distribution of nesting levels
df.group_by("NestingLevel").agg(pl.count().alias("count")).sort("NestingLevel")

/var/folders/pg/rrvt9fgx479brk1wjwzw9bqc0000gn/T/ipykernel_57096/269099081.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  df.group_by("NestingLevel").agg(pl.count().alias("count")).sort("NestingLevel")


NestingLevel,count
i64,u32
0,174358
1,3994
2,1119
3,577
4,216
5,88
6,92
7,18
8,15


In [130]:
# TODO: Find parent rows based on ParentId = SubsidiaryId
# (Commented out - we don't have SubsidiaryId column yet)

# nested_with_parents = nested.join(
#     df.select(["SubsidiaryId", "Subsidiary", "Jurisdiction"]).rename({"Subsidiary": "ParentSubsidiary", "Jurisdiction": "ParentJurisdiction"}),
#     left_on="ParentId",
#     right_on="SubsidiaryId",
#     how="left"
# )
# nested_with_parents

## 3. Rows with Footnotes

In [131]:
# Find rows with non-empty footnotes
with_footnotes = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
)

footnotes_accessions = with_footnotes.select("Accession").unique()

print(f"Rows with footnotes: {len(with_footnotes):,}")
print(f"Percentage: {len(with_footnotes) / len(df) * 100:.1f}%")

print(f"\nUnique Accession list({len(footnotes_accessions)}):")
print(footnotes_accessions.to_series().to_list())
with_footnotes.head(20)

Rows with footnotes: 1,269
Percentage: 0.7%

Unique Accession list(191):
['`000010956325000080', '`000164117225000737', '`000095017025026655', '`000095017025023207', '`000095017025020763', '`000117891325001365', '`000121390025021907', '`000092480525000012', '`000120717925000005', '`000162828025019714', '`000165495425012195', '`000132440425000006', '`000157791625000011', '`000000497725000047', '`000138865825000028', '`000095017025024783', '`000102085925000054', '`000162828025005715', '`000111505525000042', '`000190144025000012', '`000162828025037656', '`000118518525000229', '`000155837025003413', '`000182831825000057', '`000093570325000015', '`000082441025000008', '`000095017025023184', '`000171126925000004', '`000116638825000014', '`000130696525000007', '`000162828025005126', '`000121390025043459', '`000102312825000026', '`000162828025006093', '`000134287425000024', '`000008812125000017', '`000162828025008128', '`000169051125000011', '`000173399825000020', '`000141057825001050', '`0001

Accession,URL,SubsidiaryId,Subsidiary,Jurisdiction,NestingLevel,ParentName,ParentId,Ownership,Footnotes
str,str,str,str,str,i64,str,str,f64,str
"""`000149315225014286""","""https://www.sec.gov/Archives/e…","""aa9c2de8-2038-51f9-abbc-f5c42a…","""Brigadier Security Systems Ltd…","""Saskatchewan, Canada""",0,null,"""a1d1ae4e-7f40-5345-a808-c76c87…",null,"""2000"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""a2c2415d-3780-5201-877b-8d0ea7…","""DBM Global Inc.""","""Delaware""",0,null,"""ca1ae09a-5aa5-5527-931f-511e55…",91.21,"""2"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""74efcf5e-6666-5df0-8b4b-c73bdc…","""GrayWolf Industrial, Inc.""","""Delaware""",2,"""CBHorn Holdings, Inc.""","""8d35b929-5f59-5eb4-9c5e-cd7cc5…",null,"""3, 4"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""44c65a9f-c042-5cbf-ae0c-54d23c…","""GrayWolf Integrated Constructi…","""Delaware""",3,"""GrayWolf Industrial, Inc.""","""74efcf5e-6666-5df0-8b4b-c73bdc…",null,"""5"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""1d76e2b1-f60c-50fe-b3c7-ddc4c2…","""Midwest Environmental, Inc.""","""Kentucky""",3,"""GrayWolf Industrial, Inc.""","""74efcf5e-6666-5df0-8b4b-c73bdc…",null,"""6"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""e492c2df-6f8b-55f1-807c-88bcc1…","""Milco National Constructors, I…","""Delaware""",3,"""GrayWolf Industrial, Inc.""","""74efcf5e-6666-5df0-8b4b-c73bdc…",null,"""7"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""382e8b19-ebbb-5ded-be33-47f551…","""DBM Vircon Services , Inc.""","""Arizona""",2,"""DBM Global North America Inc.""","""f9c03ea7-3182-58c6-a6ec-e8bc04…",null,"""8"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""8c01e64d-96e7-5470-ad4e-bc5df2…","""Schuff Steel Company""","""Delaware""",2,"""DBM Global North America Inc.""","""f9c03ea7-3182-58c6-a6ec-e8bc04…",null,"""9"""
"""`000100683725000021""","""https://www.sec.gov/Archives/e…","""7018afcd-c2ee-58b4-ab21-c6a5c0…","""Derr and Isbell Construction, …","""Texas""",3,"""Schuff Steel Company""","""8c01e64d-96e7-5470-ad4e-bc5df2…",null,"""10"""


In [132]:
# Unique footnote values per accession
footnotes_by_accession = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
).group_by("Accession").agg(
    pl.col("Footnotes").unique().alias("UniqueFootnotes")
)

print(f"Accessions with footnotes: {len(footnotes_by_accession)}")
print(f"\nAccession: Footnotes")
for row in footnotes_by_accession.iter_rows():
    print(f"{row[0]}: {row[1]}")

Accessions with footnotes: 191

Accession: Footnotes
`000164117225000737: ['150']
`000144889325000009: ['1', '2', '3', '4', '5', '6', '7, 12', '7', '8', '9', '10', '11']
`000173399825000020: ['1']
`000005125325000013: ['1', '2', '3', '4', '5', '1990', '6', '7', '8']
`000162828025005126: ['2024']
`000162828025005715: ['43', '23', '1']
`000095017025023184: ['1']
`000077149725000031: ['2014', '2010']
`000165495425004306: ['1', '2', '3']
`000151717525000002: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18']
`000110465925121185: ['1', '2']
`000141057825001050: ['1', '2']
`000155837025002417: ['2', '3']
`000138119725000036: ['1', '2', '3']
`000009212225000018: ['2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21']
`000121465925005928: ['1']
`000095017025047918: ['2', '3', '4']
`000008812125000017: ['1']
`000143757825000007: ['1', '2']
`000121390025043484: ['1', '2']
`000159036425000006

In [133]:
# Unique footnote values - extract all individual numbers with their accessions
import re

footnotes_with_accession = df.filter(
    pl.col("Footnotes").is_not_null() & (pl.col("Footnotes").str.strip_chars() != "")
).select(["Accession", "Footnotes"]).unique()

# Build a dict: number -> list of accessions
number_to_accessions = {}
for row in footnotes_with_accession.iter_rows():
    accession, footnote = row
    nums = re.findall(r'\d+', footnote)
    for n in nums:
        num = int(n)
        if num not in number_to_accessions:
            number_to_accessions[num] = set()
        number_to_accessions[num].add(accession)

# Sort by number descending and print
sorted_items = sorted(number_to_accessions.items(), key=lambda x: x[0], reverse=True)

print(f"Total unique footnote numbers: {len(sorted_items)}")
print(f"\nFootnoteNumber: Accessions")
for num, accs in sorted_items:
    print(f"{num}: {list(accs)}")

Total unique footnote numbers: 102

FootnoteNumber: Accessions
25262: ['`000141057825001475']
25093: ['`000141057825001475']
25092: ['`000141057825001475']
2025: ['`000159036425000006']
2024: ['`000162828025005126']
2023: ['`000074026025000052']
2021: ['`000162529725000016']
2018: ['`000007747625000007']
2015: ['`000171218425000031']
2014: ['`000121390025037643', '`000077149725000031', '`000162529725000016']
2013: ['`000117891325000821']
2012: ['`000155118225000006']
2010: ['`000197413825000016', '`000077149725000031']
2009: ['`000121390025043459', '`000162828025045293']
2008: ['`000000885825000028', '`000162828025005311']
2007: ['`000162828025053207']
2005: ['`000006270925000015']
2003: ['`000162828025006093', '`000158536425000014']
2002: ['`000031920125000024']
2001: ['`000121390025020360', '`000095017025027569', '`000093796625000009']
2000: ['`000149315225014286']
1999: ['`000006270925000015']
1998: ['`000155837025001224', '`000117891325000821']
1997: ['`000102312825000026', '`00012